# core

> Scheme R5RS implementation in Python

Goal here is to incrementally build up to an implementation of Lisp in python. I'd like to use the solveit method starting from the inside out. So we'll likely start with literals and work our way up.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great. An inside-out path for a tiny Lisp in Python could start like this:

1. **Data/literals**: numbers, strings, booleans, symbols, lists
2. **Reader/parser**: turn text like `(+ 1 2)` into Python data structures
3. **Environment**: variable lookup and bindings
4. **Evaluator**: evaluate literals, symbols, and function calls
5. **Special forms**: `quote`, `if`, `define`, `lambda`
6. **Functions/macros later**: builtins, closures, maybe macros

For the very first step, I’d separate **self-evaluating literals** from **symbols**.

A minimal representation might be:

```python
class Symbol(str): pass
```

Then plain Python values can represent literals:

```python
42          # number literal
"hello"     # string literal
True        # boolean literal
Symbol("x") # Lisp symbol
```

The key distinction: `"x"` is a string value, but `Symbol("x")` means “look up the variable named x” once evaluation exists.

<details class='token-usage-details' markdown='1'><summary>$0.0048</summary>

`total=30,987 | in=30,751 | out=236 | cached=4.6% | $0.0048 | gpt-5.5`

</details>


In [ ]:
#| default_exp core

In [ ]:
#| export
from fastcore.basics import store_attr, first

In [ ]:
#| export
from compact.reader import *
from compact.types import *

### The Evaluator: eval and apply

help me understand how eval and apply work together?

In [ ]:
#| export
def scm_apply(fn, args): return fn(*args)

### Tail-Call Optimization
Naive recursive evaluation stack overflows for deep recursion (e.g. `(fact 10000)`). Instead of recursing, we can evaluate iteratively

scm_eval_one_step and scm_eval_tco recursively call each other.  


One step returns expression, if it is a thunk, it gets executed in the scm_eval_tco. This avoids having deep stacks.

In [ ]:
#| export
def scm_eval_one_step(expr, env, sfs=()):
    "eval-uate lisp expressions given an environment and handlers for special forms"
    # Atoms
    if isinstance(expr, Symbol): return env[expr.s]
    if not isinstance(expr, list): return expr

    # lists
    if expr == []: return []
    hd, *body = expr

    if isinstance(hd, Symbol) and hd.s in sfs: return sfs[hd.s](body, env, sfs)
    fn = scm_eval_tco(hd, env, sfs)
    if isinstance(fn, Macro): return Thunk(fn.fn(*body), env)

    return scm_apply(fn, [scm_eval_tco(o, env, sfs) for o in body])

In [ ]:
#| export
def scm_eval_tco(expr, env, sfs=()):
    "eval but with tail call optimization"
    while True:
        r = scm_eval_one_step(expr, env, sfs)
        if isinstance(r, Thunk): expr, env = r.expr, r.env
        else: return r

### Primitive Procedures
Give the interpreter some basic skills.

In [ ]:
#| export
import math, operator as op

def scm_sub(x, *xs): return x - sum(xs) if xs else -x
def scm_div(x, *xs): return 1/x if not xs else x / math.prod(xs)

def is_num(x): return not isinstance(x, bool) and isinstance(x, (int, float, complex))
def is_sym(x): return isinstance(x, Symbol)

_builtin = {
    "+": lambda *xs: sum(xs),
    "*": lambda *xs: math.prod(xs), 
    "/": scm_div,
    "-": scm_sub,
    ">": op.gt, 
    "<": op.lt, 
    "=": lambda x,y: is_num(x) and is_num(y) and x == y,
    "<=": op.le,
    ">=": op.ge,
    
    "number?": is_num,
    "string?": lambda x: isinstance(x, str),
    "symbol?": is_sym,
    "boolean?": lambda x: isinstance(x, bool),
    "null?": lambda x: x == [],
    "pair?": lambda x: isinstance(x, list) and len(x) > 0,
    "list?": lambda x: isinstance(x, list),

}

### Environment

In [ ]:
#| export
class Env:
    "this is an implementation of env for scheme eval()"
    def __init__(self, d, parent, primitives:dict=()): store_attr(cast=True)

    def __getitem__(self, k):
        s = self.find(k)
        if s is not None: 
            # there could be collisions for keys in primitives and globals()
            # for instance if `list` is called in scheme we want the builtin version
            # in python list will return what is in globals
            # (define list 42) is the moral equivalent of list = 42 - they will overwrite what's in globals
            if k in s.primitives: return s.primitives[k]
            return s.d[k]
        raise KeyError(k)

    def __setitem__(self, k, v): self.d[k] = v

    def __contains__(self, k): return self.find(k) is not None

    def update(self, d): self.primitives.update(d)

    def find(self, k):
        if k in self.d | self.primitives: return self
        if self.parent is not None: return self.parent.find(k)
        return None
    
    def new_frame(self, bindings=()):
        return Env(dict(bindings), parent=self)



In [ ]:
global_env = Env(globals(), parent=None, primitives=_builtin | pair_primitives)

### Special Forms

The big difference is that arguments for special forms are passed in directly without evaluation. It is up to the the special form to interpret them.

For instance:
```
(if #t 1 (/ 1 0))
```
The `else` clause should not be greedily evaluated. `if` needs to figure out if it wants to handle it.

#### quote
```lisp
    `(+ 1 2) -> (+ 1 2)
```

In [ ]:
#| export
def _sf_quote(xs, env, sfs): return list2pair(xs[0])

#### if

```lisp
    (if #t 1 (/ 1 0))
```

In [ ]:
#| export
def _sf_if_tco(xs, env, sfs):
    cond,then_,else_ = xs
    br = else_ if scm_eval_tco(cond, env, sfs) is False else then_
    return Thunk(br, env)


#### define — binding symbols

```lisp
    (define name value)

    (define square (lambda (x) (* x x)))

    (define (square x) (* x x))
```

In [ ]:
#| export
def body_expr(body): return body[0] if len(body) == 1 else [Symbol("begin"), *body]

In [ ]:
#| export
def arity(ps, vs):
    if len(vs) == len(ps): return vs
    raise ValueError(f"incorrect number of vs passed (wanted: {len(ps)}, got: {len(vs)})")

In [ ]:
#| export
def bind_params(ps, vs):
    if isinstance(ps, Symbol): return {ps.s: list(vs)}
    return dict(zip([p.s for p in ps], arity(ps, vs)))

In [ ]:
#| export
def _sf_define_tco(xs, env, sfs):
    def _mkfn(sym, expr): 
        if not isinstance(sym, Symbol): raise SyntaxError(f"{sym} must be a symbol")
        env[sym.s] = scm_eval_tco(expr, env, sfs)
        return sym.s

    arg0, *rest = xs
    if isinstance(arg0, Symbol): return _mkfn(arg0, body_expr(rest))
    if isinstance(arg0, list): return _mkfn(arg0[0], [Symbol("lambda"), arg0[1:], *rest])


we also added support for short form of define here

#### lambda — anonymous procedures

```lisp
    (lambda (x) (+ x 2))
```

In [ ]:
#| export
def _sf_lambda_tco(xs, env, sfs): 
    params, *body = xs
    def procedure(*args):
        return Thunk(body_expr(body), env.new_frame(bind_params(params, args)))
    return procedure
    # return lambda *args: Thunk(body_expr(body), Env(**bind_params(params, args), parent=env))

#### begin

```lisp
    (begin
        (define a 2)
        (+ a 2))
```

In [ ]:
#| export
def _sf_begin_tco(xs, env, sfs):
    for o in xs[:-1]: scm_eval_tco(o, env, sfs)
    return Thunk(xs[-1], env)

#### set!

In [ ]:
def _sf_set(xs, env, sfs):
    sym, expr = xs
    if not isinstance(sym, Symbol): raise SyntaxError(f"set! argument {sym} must be a symbol")
    env.find(sym.s)[sym.s] = scm_eval_tco(expr, env, sfs)
    return sym.s

In [ ]:
scm_eval_tco(parse("""
(begin
  (define x 1)
  (set! x 42)
  x)
"""), global_env, sfs={"begin": _sf_begin_tco, "define": _sf_define_tco, "set!": _sf_set})

42

In [ ]:
scm_eval_tco(
    parse("""
        (begin
          (define fact-iter
            (lambda (n acc)
            (if (= n 0) acc (fact-iter (- n 1) (* acc n)))))
          (fact-iter 5 1))
    """), 
    global_env, 
    sfs={"define":_sf_define_tco, "lambda":_sf_lambda_tco, "if":_sf_if_tco, "begin":_sf_begin_tco}
)

120

In [ ]:
scm_eval_tco(
    parse("""

(begin
  (define fact-iter
    (lambda (n acc)
    (if (= n 0) acc (fact-iter (- n 1) (* acc n)))))
  (fact-iter 1000 1))        
  
    """), 
    global_env, 
    sfs={"define":_sf_define_tco, "lambda":_sf_lambda_tco, "if":_sf_if_tco, "begin":_sf_begin_tco}
)

4023872600770937735437024339230039857193748642107146325437999104299385123986290205920442084869694048004799886101971960586316668729948085589013238296699445909974245040870737599188236277271887325197795059509952761208749754624970436014182780946464962910563938874378864873371191810458257836478499770124766328898359557354325131853239584630755574091142624174743493475534286465766116677973966688202912073791438537195882498081268678383745597317461360853795345242215865932019280908782973084313928444032812315586110369768013573042161687476096758713483120254785893207671691324484262361314125087802080002616831510273418279777047846358681701643650241536913982812648102130927612448963599287051149649754199093422215668325720808213331861168115536158365469840467089756029009505376164758477284218896796462449451607653534081989013854424879849599533191017233555566021394503997362807501378376153071277619268490343526252000158885351473316117021039681759215109077880193931781141945452572238655414610628921879602238389714760

### Primitive List Operations

In [ ]:
scm_eval_tco(parse("(car (cdr '(10 20 30)))"), global_env, sfs={"quote":_sf_quote})

20

### let — lexical bindings
```
    (let ((x 40))
      (+ x 2))

```

In [ ]:
def _sf_let(xs, env, sfs):
    binds, *body = xs
    new_env = env.new_frame({name.s: scm_eval_tco(val, env, sfs) for name, val in binds})

    return Thunk(body_expr(body), new_env)

### Macros
```
    (define unless 
      (macro (test body) `(if ,test #f ,body)))

```

In [ ]:
#| export
def _macro(args, env, sfs):
    params, *body = args
    return Macro(lambda *vals: scm_eval_tco(body_expr(body), env.new_frame(bind_params(params, vals)), sfs))

In [ ]:
#| export
def pair2list(p):
    "walk a Pair chain into a python list"
    out = []
    while isinstance(p, Pair): out.append(p.car); p = p.cdr
    return out

In [ ]:
#| export
def qq(x, env, sfs, depth=1):
    "handle ` , and ,@"
    class Matcher(Symbol):
        def match(self, x): return isinstance(x, list) and len(x) == 2 and x[0] == self
        def val(self, x): return x[1] if self.match(x) else None

    QQ, UQ, UQS = list(map(Matcher, ["quasiquote", "unquote", "unquote-splicing"]))

    if not isinstance(x, list): return x
    if QQ.match(x): return [QQ, qq(x[1], env, sfs, depth+1)]
    if UQ.match(x): return scm_eval_tco(x[1], env, sfs) if depth == 1 else [UQ, qq(x[1], env, sfs, depth-1)]
    
    if UQS.match(x):
        if depth == 1: raise SyntaxError("unquote-splicing only valid inside a list")
        return [UQS, qq(x[1], env, sfs, depth-1)]
    

    res = []
    for o in x:
        if UQS.match(o) and depth == 1: res.extend(pair2list(scm_eval_tco(o[1], env, sfs)))
        else: res.append(qq(o, env, sfs, depth))
    
    return res

In [ ]:
#| export
def _sf_quasiquote(args, env, sfs): return qq(args[0], env, sfs)

In [ ]:
scm_eval_tco(parse(
    "(let ((x 42) (xs '(1 2 3))) `(a ,x ,@xs z))"
    ), global_env,sfs={"quote":_sf_quote, "quasiquote":_sf_quasiquote, "let":_sf_let}
)

[compact.types.Symbol(s='a'), 42, 1, 2, 3, compact.types.Symbol(s='z')]

In [ ]:
scm_eval_tco(parse(
    "(begin (define unless (macro (test body) `(if ,test #f ,body))) (unless (= 1 2) 42))"
    ), global_env, sfs={"quasiquote":_sf_quasiquote, "macro":_macro, "define":_sf_define_tco, "if":_sf_if_tco, "begin":_sf_begin_tco})

42

In [ ]:
try: parse("'(. 1)")
except SyntaxError: pass

In [ ]:
try: parse("'(1 . 2 3)")
except SyntaxError: pass

Take a look at the dialogs in this project and let me know what the gaps are from SICP.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I'll scan the project notebooks/dialogs at a high level first — mainly filenames and headings — then compare what’s already covered against the main SICP Scheme/interpreter ideas.



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from rgapi.skill import fd, nbrg, rg\nfd(…&quot;)→&quot;[&#x27;CLAUDE_MD.ipynb&#x27;, &#x27;nbs/index.ipynb&#x27;, &#x27;…&quot;</code></summary>

```json
{
  "id": "fc_0a9c5b914bd08b0e016a6c242be7b88190a9cca0ae495cdb33",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from rgapi.skill import fd, nbrg, rg\nfd('/app/data/projects/compact', ext='ipynb')"
    }
  },
  "result": "['CLAUDE_MD.ipynb', 'nbs/index.ipynb', 'nbs/01_types.ipynb', 'nbs/00_core.ipynb', 'nbs/02_reader.ipynb', 'nbdev3-template-0.0.3/index.ipynb', 'nbdev3-template-0.0.3/00_core.ipynb']"
}
```

</details>





I found the project notebooks: `00_core`, `01_types`, `02_reader`, plus `index` and a template copy. Next I’ll look at headings/structure in the actual `nbs/` dialogs.



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;print(nbrg(r&#x27;&#x27;&#x27;^#{1,4} |^def |^class &#x27;&#x27;&#x27;…&quot;)→&quot;00_core.ipynb:277a059d:# core\\n\\n&gt; Schem…&quot;</code></summary>

```json
{
  "id": "fc_04cd824185bd6116016a6c2431451c81979378ae4332989ec1",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "print(nbrg(r'''^#{1,4} |^def |^class ''', root='/app/data/projects/compact/nbs', include='*.ipynb'))"
    }
  },
  "result": "<TRUNCATED>…ynb:277a059d:# core\\n\\n> Scheme R5RS implementation in Python\n00_core.ipynb:0ce25a3e:Goal here is to incrementally build up to an implementation of Lisp in python. I'd like to use the solveit method starti…\n00_core.ipynb:54be8fdd:### The Evaluator: eval and apply\n00_core.ipynb:48103d34:#| export\\ndef scm_apply(fn, args): return fn(*args)\n00_core.ipynb:c02091d6:### Tail-Call Optimization\\nNaive recursive evaluation stack overflows for deep recursion (e.g. `(fact 10000)`). Instead…\n00_core.ipynb:fd519e55:#| export\\ndef scm_eval_one_step(expr, env, sfs=()):\\n    \"eval-uate lisp expressions given an environment and handlers …\n00_core.ipynb:12289072:#| export\\ndef scm_eval_tco(expr, env, sfs=()):\\n    \"eval but with tail call optimization\"\\n    while True:\\n        r …\n00_core.ipynb:2dc40a55:### Primitive Procedures\\nGive the interpreter some basic skills.\n00_core.ipynb:5c4b439c:#| export\\nimport math, operator as op\\n\\ndef scm_sub(x, *xs): return x - sum(xs) if xs else -x\\ndef scm_div(x, *xs): re…\n00_core.ipynb:b8d168b4:### Environment\n00_core.ipynb:24df7be0:#| export\\nclass Env:\\n    \"this is an implementation of env for scheme eval()\"\\n    def __init__(self, d, parent, primi…\n00_core.ipynb:a29b6976:### Special Forms\\n\\nThe big difference is that arguments for special forms are passed in directly without evaluation. I…\n00_core.ipynb:f7811aa3:#### quote\\n```lisp\\n    `(+ 1 2) -> (+ 1 2)\\n```\n00_core.ipynb:cf72bbe6:#| export\\ndef _sf_quote(xs, env, sfs): return list2pair(xs[0])\n00_core.ipynb:ed04ec82:#### if\\n\\n```lisp\\n    (if #t 1 (/ 1 0))\\n```\n00_core.ipynb:f72f6bd8:#| export\\ndef _sf_if_tco(xs, env, sfs):\\n    cond,then_,else_ = xs\\n    br = else_ if scm_eval_tco(cond, env, sfs) is F…\n00_core.ipynb:e697c273:#### define — binding symbols\\n\\n```lisp\\n    (define name value)\\n\\n    (define square (lambda (x) (* x x)))\\n\\n    (de…\n00_core.ipynb:a1c91964:#| export\\ndef body_expr(body): return body[0] if len(body) == 1 else [Symbol(\"begin\"), *body]\n00_core.ipynb:efce91e6:#| export\\ndef arity(ps, vs):\\n    if len(vs) == len(ps): return vs\\n    raise ValueError(f\"incorrect number of vs passe…\n00_core.ipynb:e357365b:#| export\\ndef bind_params(ps, vs):\\n    if isinstance(ps, Symbol): return {ps.s: list(vs)}\\n    return dict(zip([p.s fo…\n00_core.ipynb:cf48dc77:#| export\\ndef _sf_define_tco(xs, env, sfs):\\n    def _mkfn(sym, expr): \\n        if not isinstance(sym, Symbol): raise …\n00_core.ipynb:7382a506:#### lambda — anonymous procedures\\n\\n```lisp\\n    (lambda (x) (+ x 2))\\n```\n00_core.ipynb:d872eca0:#| export\\ndef _sf_lambda_tco(xs, env, sfs): \\n    params, *body = xs\\n    def procedure(*args):\\n        return Thunk(b…\n00_core.ipynb:74759d81:#### begin\\n\\n```lisp\\n    (begin\\n        (define a 2)\\n        (+ a 2))\\n```\n00_core.ipynb:578f1427:#| export\\ndef _sf_begin_tco(xs, env, sfs):\\n    for o in xs[:-1]: scm_eval_tco(o, env, sfs)\\n    return Thunk(xs[-1], e…\n00_core.ipynb:5983a4af:#### set!\n00_core.ipynb:d3431242:def _sf_set(xs, env, sfs):\\n    sym, expr = xs\\n    if not isinstance(sym, Symbol): raise SyntaxError(f\"set! argument {s…\n00_core.ipynb:789a37bc:### Primitive List Operations\n00_core.ipynb:edde9c33:### let — lexical bindings\\n```\\n    (let ((x 40))\\n      (+ x 2))\\n\\n```\n00_core.ipynb:8746edea:def _sf_let(xs, env, sfs):\\n    binds, *body = xs\\n    new_env = env.new_frame({name.s: scm_eval_tco(val, env, sfs) for …\n00_core.ipynb:cf7b939f:### Macros\\n```\\n    (define unless \\n      (macro (test body) `(if ,test #f ,body)))\\n\\n```\n00_core.ipynb:7300b953:#| export\\ndef _macro(args, env, sfs):\\n    params, *body = args\\n    return Macro(lambda *vals: scm_eval_tco(body_expr(…\n00_core.ipynb:8ae70d8c:#| export\\ndef pair2list(p):\\n    \"walk a Pair chain into a python list\"\\n    out = []\\n    while isinstance(p, Pair): o…\n00_core.ipynb:f3037817:#| export\\ndef qq(x, env, sfs, depth=1):\\n    \"handle ` , and ,@\"\\n    class Matcher(Symbol):\\n        def match(self, x…\n00_core.ipynb:78732c31:#| export\\ndef _sf_quasiquote(args, env, sfs): return qq(args[0], env, sfs)\n00_core.ipynb:82316072:### nbdev postscript\n01_types.ipynb:31dbc74d:# types\\n\\n> Scheme types\n01_types.ipynb:edded0fb:### nbdev prologue\n01_types.ipynb:bcd6a16c:#| export\\nclass Symbol:\\n    \"symbol is a `name` in Lisp\"\\n    def __init__(self, s): store_attr()\\n    def __str__(sel…\n01_types.ipynb:2a368749:#| export\\nclass Macro:\\n    \"holder of the function for late evaluation\"\\n    def __init__(self, fn): store_attr()\n01_types.ipynb:66d47471:#| export\\nclass Thunk:\\n    \"a piece of computation that can be executed later\"\\n    def __init__(self, expr, env): sto…\n01_types.ipynb:a142753a:#| export\\nclass Pair:\\n    \"a cons cell\"\\n    __slots__ = (\"car\", \"cdr\")\\n    def __init__(self, car, cdr): self.car,se…\n01_types.ipynb:f8c1dae5:#| export\\nclass NilType:\\n    def __str__(self): return \"()\"\\n    def __repr__(self): return str(self)\n01_types.ipynb:08267157:#| export\\ndef list2pair(lst, tail=Nil):\\n    \"convert a python list into Scheme list /…</TRUNCATED>"
}
```

</details>





I scanned the project dialogs/notebooks and the current implementation already has a solid “small Scheme core”:

- `01_types`: `Symbol`, `Pair`, `Nil`, `Thunk`, `Macro`
- `02_reader`: atoms, lexer/parser, quote syntax, dotted pairs, `parse_all`
- `00_core`: `eval`/`apply`, TCO, env frames, primitives, `quote`, `if`, `define`, `lambda`, `begin`, `set!`, `let`, macros, quasiquote/unquote/unquote-splicing

The main gaps from a SICP-style Scheme journey are roughly:

1. **More special forms**
   - `cond`
   - `and`, `or`
   - `let*`, `letrec`
   - maybe named `let`
   - `delay` / `force` for streams

2. **More primitive procedures**
   - list helpers: `cons`, `car`, `cdr`, `list`, `append`, `map`, `filter`, `length`, `reverse`
   - predicates: `eq?`, `eqv?`, `equal?`
   - numeric helpers: `abs`, `remainder`, `modulo`, `zero?`, `positive?`, `negative?`
   - Scheme-level `apply`

3. **Evaluator polish from SICP Chapter 4**
   - proper procedure objects instead of raw Python closures, if you want closer SICP structure
   - internal definitions / `scan-out-defines`
   - derived-expression transforms, e.g. implement `cond` by rewriting to `if`
   - better error messages for malformed forms
   - REPL / driver loop

4. **Data abstraction examples from SICP Chapter 2**
   - rational numbers
   - points/segments
   - symbolic differentiation
   - trees
   - sets
   - Huffman encoding  
   
   These are not interpreter features exactly, but they are central SICP exercises that test whether the Lisp is pleasant to use.

5. **State, mutation, and streams from Chapter 3**
   - `set-car!`, `set-cdr!`
   - mutable pairs/lists semantics
   - stream support with delayed evaluation
   - examples like queues, tables, memoization, constraint propagation

6. **Big optional later topics**
   - lazy evaluator
   - nondeterministic evaluator with `amb`
   - logic programming/query evaluator
   - register-machine simulator
   - compiler / explicit-control evaluator

If you want the next “inside out” step, I’d probably fill in **derived special forms** next: `cond`, `and`, `or`, `let*`, and `letrec`. They build directly on the evaluator you already have, and they’re very SICP-aligned without requiring a big redesign.

<details class='token-usage-details' markdown='1'><summary>$0.0182</summary>

`total=114,074 | in=112,245 | out=1,829 | cached=70.4% | reasoning=1,051 | $0.0182 | gpt-5.5`

</details>


### nbdev postscript

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()